# Squad optimization (2024-25)

Picks our squad each gameweek, rolls that decision through the season and compares it with the
real history. Two optimizers are run, each over a whole season:

* **base**: the squad, XI and captain with the most expected points ($\lambda = 0$), within the
  budget and free transfers; it transfers whenever that adds expected points.
* **MVT**: the mean-variance squad that maximises the probability of beating that week's
  head-to-head opponent (steps 1-3 below); it transfers only if that beats holding by
  `transfer_margin`.

Both are scored against the same opponent squads, so the base optimizer's $P(\text{beat})$ is
reported too.

1. **Opponent squads**: sample the opponent's likely squads $\mathbb{W}_o$ from their selection
   model (league betas + their manager fit) with a sparse C-vine copula.
2. **Points distribution**: $\mu_\delta$ is the predictor's xP for the gameweek and $\Sigma_\delta$ the
   covariance of its past residuals (actual − predicted xP).
3. **Mean-variance MIQP**: for each $\lambda$, choose a 15-player squad within budget and free
   transfers; keep the $\lambda$ with the highest Monte Carlo probability of beating the opponent's
   $r$-th best sampled squad, and only transfer if that beats holding by `transfer_margin`.
4. **Rolling season**: carry the simulated squad, free-transfer bank and money in the bank from
   week to week.

The swap sampler for $\mathbb{W}_o$ (the opponent's current squad plus keep/buy swaps) is compared
with the C-vine in `optimization_swap.ipynb`, outside the thesis.

Decisions for gameweek $t$ only use data up to $t-1$ (features, ownership and prices from the
previous round), as in `copula_estimation.ipynb`.

**Inputs** (paths relative to this notebook)

| What | Path | Built by |
|---|---|---|
| Player data, every player each round | `../inputs/rolled_data_24-25_xp_fixed.csv` | `optimizer/prepare_data.py` |
| League squads | `../inputs/league_selections_df.csv` | |
| League betas | `../beta estimators/league betas/league_models_{pos}` | `beta_{pos}.ipynb` |
| Manager fits | `../beta estimators/opponent betas/estimates/manager_fits_{GW}.joblib` | `longitudinal.ipynb` |
| xP predictions | `../../predictor/models/predictions.csv` | `predictor/src/predictor.py` |

Step 3 uses SCIP through `pyscipopt` (`pip install pyscipopt`; free, no licence needed).

## Setup

In [1]:
import ast
import random

import joblib
import numpy as np
import pandas as pd
import pyvinecopulib as pv
from scipy.optimize import minimize_scalar
from scipy.special import expit, logsumexp
from scipy.stats import kendalltau
from sklearn.linear_model import LogisticRegression

from pyscipopt import Model, quicksum

# Inputs
PLAYER_DATA_PATH = "../inputs/rolled_data_24-25_xp_fixed.csv"
LEAGUE_SELECTIONS_PATH = "../inputs/league_selections_df.csv"
LEAGUE_FITS_PATH = "../beta estimators/league betas/league_models_{pos}"
MANAGER_FITS_PATH = (
    "../beta estimators/opponent betas/estimates/manager_fits_{GW}.joblib"
)
PREDICTIONS_PATH = "../../predictor/models/predictions.csv"
# Sigma_delta: "structured" (team factor fitted to correlations by position, sigma.py) or
# "sample" (per-pair sample covariance of the residuals, the saved runs)
SIGMA = "structured"

# Output
ROLLING_RESULTS_PATH = (
    "./rolling_optimization_results_{method}"
    + ("" if SIGMA == "sample" else "_structured")
    + ".joblib"
)
# The optimizers compared, one rolling season each: "base" (most expected points, lambda = 0)
# and "mvt" (most probable win against the opponent, over the lambda grid)
METHODS = ["base", "mvt"]

POSITIONS = ["Goalkeeper", "Defender", "Midfielder", "Forward"]
POSITION_SHORT = {
    "Goalkeeper": "gk",
    "Defender": "def",
    "Midfielder": "mid",
    "Forward": "fwd",
}
POSITION_BINARY_COL = {
    "Goalkeeper": "goalkeeper_binary",
    "Defender": "defender_binary",
    "Midfielder": "midfielder_binary",
    "Forward": "forward_binary",
}
SQUAD_SLOTS = {"Goalkeeper": 2, "Defender": 5, "Midfielder": 5, "Forward": 3}

# League
OWN_TEAM_ID = 205  # "us" throughout this whole project
MANAGERS = [
    205,
    901,
    4213,
    7497,
    13603,
    23881,
    38054,
    38351,
    69718,
    152245,
    198971,
    577273,
    593532,
    696127,
    749244,
    1222335,
    1441046,
    1546929,
    2634457,
    4131447,
]
FIXTURE_SEEDS = (61, 16)  # first and second half of the season

In [2]:
player_data = pd.read_csv(PLAYER_DATA_PATH, low_memory=False)
league_selections = pd.read_csv(LEAGUE_SELECTIONS_PATH)
LEAGUE_FITS = {
    pos: joblib.load(LEAGUE_FITS_PATH.format(pos=short))
    for pos, short in POSITION_SHORT.items()
}
print(
    "league fits for gameweeks",
    {pos: f"{min(f)}-{max(f)}" for pos, f in LEAGUE_FITS.items()},
)

league fits for gameweeks {'Goalkeeper': '6-38', 'Defender': '6-38', 'Midfielder': '6-38', 'Forward': '6-38'}


## League fixtures

A single round-robin per half of the season, the same draw as `longitudinal.ipynb`.

In [3]:
def create_fixtures(managers, seed=None):
    """Randomized single round-robin fixture (verbatim from longitudinal.ipynb)."""
    if seed is not None:
        random.seed(seed)
    managers = managers.copy()
    random.shuffle(managers)
    fixed = managers[0]
    rotating = managers[1:]
    fixtures = {}
    for gw in range(1, len(managers)):
        current = [fixed] + rotating
        gw_fixtures = {}
        for i in range(len(current) // 2):
            m1, m2 = current[i], current[-(i + 1)]
            gw_fixtures[m1] = m2
            gw_fixtures[m2] = m1
        fixtures[gw] = gw_fixtures
        rotating = [rotating[-1]] + rotating[:-1]
    return fixtures


def get_opponent(manager_id, gameweek, fixtures):
    return fixtures[gameweek][manager_id]


fixtures_leg_1 = create_fixtures(MANAGERS, seed=FIXTURE_SEEDS[0])
fixtures_leg_2 = create_fixtures(MANAGERS, seed=FIXTURE_SEEDS[1])
opponents_by_round = {
    gw: get_opponent(OWN_TEAM_ID, gw, fixtures_leg_1) for gw in range(1, 20)
} | {gw + 19: get_opponent(OWN_TEAM_ID, gw, fixtures_leg_2) for gw in range(1, 20)}
print({gw: opponents_by_round[gw] for gw in range(6, 39)})

{6: 23881, 7: 38054, 8: 13603, 9: 593532, 10: 749244, 11: 696127, 12: 4131447, 13: 152245, 14: 1441046, 15: 1222335, 16: 1546929, 17: 198971, 18: 2634457, 19: 577273, 20: 69718, 21: 4213, 22: 1441046, 23: 4131447, 24: 1222335, 25: 152245, 26: 7497, 27: 901, 28: 198971, 29: 749244, 30: 593532, 31: 23881, 32: 1546929, 33: 577273, 34: 2634457, 35: 696127, 36: 38351, 37: 38054, 38: 13603}


## Squad history: transfers, ownership and player details

In [4]:
def detect_squad_overhaul(team_id, gw, league_selections, threshold=8):
    """
    Flags gameweek `gw` as a likely Wildcard/Free Hit week for `team_id` (see
    longitudinal.ipynb): more than `threshold` of the 15 squad slots changed from
    gw-1's actual squad is the signature of a chip-driven full rebuild, which
    the selection model has no signal for. Returns (n_changed, is_overhaul); both None if
    gw-1 or gw isn't in league_selections for this team.
    """
    prev_rows = league_selections[
        (league_selections["team_id"] == team_id)
        & (league_selections["round"] == gw - 2)
    ]
    cur_rows = league_selections[
        (league_selections["team_id"] == team_id)
        & (league_selections["round"] == gw - 1)
    ]
    if len(prev_rows) == 0 or len(cur_rows) == 0:
        return None, None
    prev_squad = set(ast.literal_eval(prev_rows.iloc[0]["squad"]))
    cur_squad = set(ast.literal_eval(cur_rows.iloc[0]["squad"]))
    n_changed = len(cur_squad - prev_squad)
    return n_changed, n_changed > threshold


def compute_free_transfers_available(
    team_id, target_round, league_selections, cap=5, overhaul_threshold=8
):
    """
    Simulates `team_id`'s banked free transfers (FTs) entering `target_round`, from
    FPL's accrual rule: 1 new FT per week, unused ones roll over up to `cap` banked at
    once; making more transfers than banked forces a points hit and drains the bank to
    0 (so next week starts back at the baseline 1). GW1 is initial squad formation (no
    transfer concept), so the bank starts at 1 entering GW2.

    Wildcard/Free Hit weeks (flagged by detect_squad_overhaul) are transparent to the
    bank: neither consumes nor resets it, so that week's actual squad-composition
    change is ignored here and the bank just accrues its usual +1 (capped).

    This uses only observed squad snapshots (league_selections), so it can look back
    to GW1 regardless of `min_round` elsewhere in the pipeline (model features aren't
    needed here).
    """
    ft_bank = 1
    for w in range(2, target_round):
        n_changed, is_overhaul = detect_squad_overhaul(
            team_id, w, league_selections, threshold=overhaul_threshold
        )
        if n_changed is None:
            continue
        if is_overhaul:
            ft_bank = min(cap, ft_bank + 1)
        elif n_changed <= ft_bank:
            ft_bank = min(cap, (ft_bank - n_changed) + 1)
        else:
            ft_bank = 1
    return ft_bank


def get_owned(team_id, position, rnd, elements):
    """Ownership map {element: 0/1} for `position` at round `rnd`, or None if that
    round's selection export doesn't line up with `elements`."""
    rows = league_selections[
        (league_selections["team_id"] == team_id) & (league_selections["round"] == rnd)
    ]
    if len(rows) == 0:
        return None
    y = np.asarray(
        ast.literal_eval(rows.iloc[0][POSITION_BINARY_COL[position]]), dtype=float
    )
    if len(y) != len(elements):
        return None
    return dict(zip(elements.tolist(), y.tolist()))


# every player's position (no player changes position during the season)
PLAYER_POSITION = (
    player_data.drop_duplicates("element").set_index("element")["position"].to_dict()
)


def squad_players(team_id, rnd):
    """The 15 players in `team_id`'s round-`rnd` squad, or None if that round is missing.
    From the `squad` column, not the per-position binary columns: those only cover the
    players with a row that round, so a player without a fixture (postponed matches in
    GW15, 29 and 34) drops out of them and 40 squads come out with 11 to 14 players. In
    Assistant Manager weeks the `squad` column carries a 16th element that isn't a player
    (e.g. element 742 at GW25), dropped here as it is in no round's data."""
    rows = league_selections[
        (league_selections["team_id"] == team_id) & (league_selections["round"] == rnd)
    ]
    if len(rows) == 0:
        return None
    return {e for e in ast.literal_eval(rows.iloc[0]["squad"]) if e in PLAYER_POSITION}


def squad_in_position(team_id, rnd, position):
    """The `position` players of squad_players(team_id, rnd) (empty if the round is missing)."""
    return {
        e for e in squad_players(team_id, rnd) or () if PLAYER_POSITION[e] == position
    }


def describe_squad(elements, gw):
    """Map a squad's element IDs to readable {element, name, team, position} dicts,
    using the price/position table for `gw` (falls back to the most recent round at or
    before `gw` if a player has no row exactly at `gw`, e.g. injury/loan gaps)."""
    rows = (
        player_data[player_data["round"] == gw]
        .drop_duplicates("element")
        .set_index("element")
    )
    fallback = (
        player_data[player_data["round"] <= gw]
        .sort_values("round")
        .drop_duplicates("element", keep="last")
        .set_index("element")
    )
    out = []
    for e in elements:
        if e in rows.index:
            r = rows.loc[e]
        elif e in fallback.index:
            r = fallback.loc[e]
        else:
            out.append({"element": e, "name": None, "team": None, "position": None})
            continue
        out.append(
            {
                "element": e,
                "name": r["name"],
                "team": r["team"],
                "position": r["position"],
            }
        )
    return out

## Selection model: who a manager is likely to pick

$\alpha$ combines the league betas with the manager's own deviation, retention term and
intercept (Dirichlet-Multinomial fits). The keep/buy step (eq. 3.6) turns the linear
predictor into calibrated $P(\text{player is one of the } k \text{ picked})$: each player
owned last week is kept with a logistic probability, and the remaining slots go to everyone
else by a sharpened softmax, both fit on the whole league's history before the gameweek.

In [5]:
def get_league_fit_for_round(league_fits, target_round):
    """Pick the league fit at `target_round`, or the most recent one available
    before it (see longitudinal.ipynb for the full rationale)."""
    available = [gw for gw in league_fits if gw <= target_round]
    if not available:
        raise ValueError(f"No league fit available at or before round {target_round}")
    return league_fits[max(available)]


def _position_pool(player_data, position, rnd, feats):
    """The player pool for `position` at gameweek `rnd`, sorted by element id
    ascending -- matches the *_binary columns in league_selections for that round."""
    pool = player_data[
        (player_data["round"] == rnd) & (player_data["position"] == position)
    ].sort_values("element")
    elements = pool["element"].to_numpy()
    X_raw = pool[feats].fillna(0).to_numpy(dtype=float)
    return elements, X_raw


def alpha_manager(manager_fit, league_fit, X_new_raw, y_prev):
    """alpha_{j,m,t,k} (eq:alpha_m) for a fitted manager model at new raw features
    X_new_raw and retention lag y_prev."""
    X_std_new = (X_new_raw - league_fit["X_mean"]) / league_fit["X_std"]
    beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
    linear = (
        manager_fit.get("intercept_mean", 0.0)
        + X_std_new @ beta_m
        + manager_fit["gamma_mean"] * y_prev
    )
    linear = np.clip(linear, -30, 30)
    return np.exp(linear)


def marginal_inclusion_probabilities(weights, k, rng, n_sim=3000):
    """
    Efraimidis-Spirakis (A-ES) weighted sampling without replacement: draw
    key_i = U_i^(1/w_i) for U_i~Uniform(0,1), take the k items with the largest keys.
    This reproduces exactly sequential (Plackett-Luce) weighted sampling without
    replacement, and by linearity of expectation the resulting per-item inclusion
    probabilities always sum to exactly k -- turning `weights` (alpha_manager's raw,
    arbitrarily-scaled output) into genuine marginal "P(this player is one of the k
    picked)" probabilities. See copula_estimation.ipynb for why this replaces the naive
    alpha/alpha.sum() (which sums to 1 per position, not k).
    """
    weights = np.clip(np.asarray(weights, dtype=float), 1e-12, None)
    n = len(weights)
    U = rng.uniform(size=(n_sim, n))
    keys = U ** (1.0 / weights)[None, :]
    topk_idx = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
    counts = np.bincount(topk_idx.ravel(), minlength=n)
    return counts / n_sim


def position_alpha_at_week(team_id, position, manager_fit, league_fit, rnd):
    """Raw alpha_{j,m,rnd,k} (eq:alpha_m) for every player in `position`'s pool at
    week `rnd`, plus that week's ownership map (or None if unavailable). Used both as
    the weights for marginal_inclusion_probabilities and, later, as the weights for
    keying the vine's correlated draws when constructing full candidate squads."""
    elements, X_raw = _position_pool(
        player_data, position, rnd - 1, league_fit["feats"]
    )
    # recompute the previous week's pool fresh -- it's a different size than `elements`
    # most weeks, so reusing `elements` here would silently zero out y_prev
    prev_elements, _ = _position_pool(
        player_data, position, rnd - 2, league_fit["feats"]
    )
    owned_prev_map = get_owned(team_id, position, rnd - 2, prev_elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    owned_map = get_owned(team_id, position, rnd - 1, elements)
    return dict(zip(elements.tolist(), alpha.tolist())), owned_map


def position_week_design(team_id, position, league_fit, rnd):
    """Same week alignment as position_alpha_at_week: the pool and ownership at
    round rnd-1, the retention lag from round rnd-2 (its own pool, see markdown above),
    and the features standardized on `league_fit`'s training mean/std."""
    elements, X_raw = _position_pool(
        player_data, position, rnd - 1, league_fit["feats"]
    )
    prev_elements, _ = _position_pool(
        player_data, position, rnd - 2, league_fit["feats"]
    )
    owned_prev_map = get_owned(team_id, position, rnd - 2, prev_elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    owned_map = get_owned(team_id, position, rnd - 1, elements)
    X_std = (X_raw - league_fit["X_mean"]) / league_fit["X_std"]
    return elements, X_std, y_prev, owned_map


def keep_buy_marginals(eta, y_prev, k, calib):
    """eq. 3.6: P(kept) = logit^-1(a + b*eta) for last week's squad; the expected
    number of buys (k - sum of keep probabilities) is spread over everyone else by
    softmax(tau * eta). Sums to k (up to the cap at 1)."""
    p = np.zeros(len(eta))
    owned = y_prev == 1
    p_keep = expit(calib["a"] + calib["b"] * eta[owned])
    p[owned] = p_keep
    if (~owned).any():
        n_buy = max(k - p_keep.sum(), 0.0)
        z = calib["tau"] * eta[~owned]
        p[~owned] = np.minimum(n_buy * np.exp(z - logsumexp(z)), 1.0)
    return p


def fit_keep_buy_calibration(position, league_fit, max_round, min_round=5):
    """
    Maximum-likelihood (a, b, tau) for keep_buy_marginals, pooled over every league
    manager's non-overhaul weeks rnd in [min_round, max_round) -- pass the same cutoff
    as the history window so nothing from the target week leaks in. Uses the
    league-level eta = beta^league . X (see copula_estimation.ipynb). min_round=5 keeps round
    rnd-1 inside the league fits' training window (rounds > 3). Returns None when there
    is no such week yet (the first gameweeks of the season); tau is 1 until a buy has
    been seen in the position.
    """
    keep_eta, keep_x, buy_groups = [], [], []
    for team_id in league_selections["team_id"].unique():
        for rnd in range(min_round, max_round):
            if detect_squad_overhaul(team_id, rnd, league_selections)[1]:
                continue
            elements, X_std, y_prev, owned_map = position_week_design(
                team_id, position, league_fit, rnd
            )
            if owned_map is None or y_prev.sum() == 0:
                continue
            x = np.array([owned_map[e] for e in elements])
            eta = X_std @ league_fit["beta_mean"]
            owned = y_prev == 1
            keep_eta.append(eta[owned])
            keep_x.append(x[owned])
            if x[~owned].sum() > 0:
                buy_groups.append((eta[~owned], x[~owned]))

    if not keep_eta:
        # too early in the season: no earlier week to learn keeping from
        return None
    keep_eta, keep_x = np.concatenate(keep_eta), np.concatenate(keep_x)
    if 0 < keep_x.mean() < 1:
        keep_model = LogisticRegression(C=1.0).fit(keep_eta[:, None], keep_x)
        a, b = float(keep_model.intercept_[0]), float(keep_model.coef_[0, 0])
    else:  # nobody ever sold (or kept): constant keep probability
        rate = np.clip(keep_x.mean(), 1e-3, 1 - 1e-3)
        a, b = float(np.log(rate / (1 - rate))), 0.0

    def buy_nll(tau):
        # conditional logit: each observed buy is a softmax(tau * eta) draw over the
        # players not owned last week
        return -sum(
            (x * tau * eta).sum() - x.sum() * logsumexp(tau * eta)
            for eta, x in buy_groups
        )

    if buy_groups:
        tau = minimize_scalar(buy_nll, bounds=(0.01, 50.0), method="bounded").x
    else:
        # no buys seen yet in this position (rare early on, e.g. goalkeepers): leave
        # eta unsharpened rather than dropping the whole calibration
        tau = 1.0
    return {
        "a": a,
        "b": b,
        "tau": float(tau),
        "n_keep_obs": len(keep_x),
        "n_buy_weeks": len(buy_groups),
    }


def position_marginals_at_week(
    team_id, position, manager_fit, league_fit, rnd, rng, calib=None
):
    """p_{k,rnd}, the marginal probability player k is one of position's squad slots
    that week, plus that week's ownership map (or None if unavailable). With `calib`
    (from fit_keep_buy_calibration) these are the keep/buy marginals of eq. 3.6;
    without it, position_alpha_at_week's weights rescaled to selection probabilities
    via marginal_inclusion_probabilities (see copula_estimation.ipynb)."""
    if calib is not None:
        elements, X_std, y_prev, owned_map = position_week_design(
            team_id, position, league_fit, rnd
        )
        eta = X_std @ (league_fit["beta_mean"] + manager_fit["delta_beta_mean"])
        p = keep_buy_marginals(eta, y_prev, SQUAD_SLOTS[position], calib)
        return dict(zip(elements.tolist(), p.tolist())), owned_map

    alpha_map, owned_map = position_alpha_at_week(
        team_id, position, manager_fit, league_fit, rnd
    )
    elements = np.array(list(alpha_map.keys()))
    alpha = np.array(list(alpha_map.values()))
    p = marginal_inclusion_probabilities(alpha, SQUAD_SLOTS[position], rng)
    return dict(zip(elements.tolist(), p.tolist())), owned_map


def squad_prices(GW):
    """{element: price} for squads entering round GW: each player's latest price at or
    before GW-1 (so a squad member without a GW-1 fixture keeps his price), or round GW's
    for a newcomer."""
    rows = player_data[player_data["round"] <= GW].sort_values("round")
    prices = rows.drop_duplicates("element", keep="last").set_index("element")["value"]
    prices = prices.to_dict()
    prices.update(
        rows[rows["round"] <= GW - 1]
        .drop_duplicates("element", keep="last")
        .set_index("element")["value"]
        .to_dict()
    )
    return prices


def position_alpha_target(team_id, position, manager_fit, league_fit, GW):
    """
    alpha for the target week GW itself: round GW's features (lagged, so known before the
    deadline) and the GW-1 squad as the retention lag -- the pairing the fits learn from
    (week t's features and week t-1's squad for week t's picks).
    position_alpha_at_week(rnd) is a week behind (features
    rnd-1, lag rnd-2), which is right for the history weeks, where the rnd-1 squad is the
    outcome; at the target week it would predict the GW-1 squad, and after a wildcard in
    GW-1 put almost no weight on the squad the manager holds.

    Squad members without a round GW row (no fixture) are scored on their latest earlier
    features. Returns ({element: alpha}, the position's GW-1 squad players).
    """
    owned = squad_in_position(team_id, GW - 1, position)
    elements, X_raw = _position_pool(player_data, position, GW, league_fit["feats"])
    missing = sorted(owned - set(elements.tolist()))
    if missing:
        latest = (
            player_data[
                (player_data["round"] < GW) & player_data["element"].isin(missing)
            ]
            .sort_values("round")
            .drop_duplicates("element", keep="last")
        )
        elements = np.concatenate([elements, latest["element"].to_numpy()])
        X_raw = np.vstack(
            [X_raw, latest[league_fit["feats"]].fillna(0).to_numpy(dtype=float)]
        )
    y_prev = np.array([1.0 if e in owned else 0.0 for e in elements])
    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    return dict(zip(elements.tolist(), alpha.tolist())), owned


## Opponent squads: C-vine sampler

`run_copula_pipeline` builds $\mathbb{W}_o$ with a sparse C-vine copula over the players with
ownership history: latent uniforms are sampled from the vine and keyed by $\alpha$ into full
squads, under the opponent's free transfers and squad value. On this data its fitted dependence
reduces to independence (diagnostics in `copula_estimation.ipynb`).


In [6]:
def latent_uniforms(X_hist, P_hist, rng):
    a = np.where(X_hist.values == 0, 0.0, 1.0 - P_hist.values)
    b = np.where(X_hist.values == 0, 1.0 - P_hist.values, 1.0)
    draws = rng.uniform(size=X_hist.shape)
    U = a + draws * (b - a)
    return pd.DataFrame(U, index=X_hist.index, columns=X_hist.columns)


# Sparse vine settings. The original setup (6 families, AIC, 15 trees over all ~120 pool
# players) fit as many non-independent pairs on column-permuted data (dependence
# destroyed) as on the real data, and scored far below independence on held-out weeks.
# Gaussian-only + BIC + one tree + a |tau| threshold is the only configuration tried
# whose in-sample fit separates from that permutation null (see `diagnostics` below).
BICOP_FAMILY_SET = [pv.BicopFamily.indep, pv.BicopFamily.gaussian]
SELECTION_CRITERION = "bic"
TRUNC_LVL = 1
TAU_THRESHOLD = 0.35


def kendall_cvine_order(U_vals):
    """C-vine root order (1-indexed) by descending hub score S_k = sum_j |tau_kj|."""
    m = U_vals.shape[1]
    tau_matrix = np.zeros((m, m))
    for i in range(m):
        for j in range(i + 1, m):
            tau, _ = kendalltau(U_vals[:, i], U_vals[:, j])
            tau = 0.0 if np.isnan(tau) else tau
            tau_matrix[i, j] = tau_matrix[j, i] = tau
    hub_scores = np.abs(tau_matrix).sum(axis=1)
    return (np.argsort(-hub_scores) + 1).tolist()


def fit_sparse_cvine(U_vals):
    cvine_structure = pv.CVineStructure(
        order=kendall_cvine_order(U_vals), trunc_lvl=TRUNC_LVL
    )
    cvine_controls = pv.FitControlsVinecop(
        family_set=BICOP_FAMILY_SET,
        selection_criterion=SELECTION_CRITERION,
        trunc_lvl=TRUNC_LVL,
        threshold=TAU_THRESHOLD,
    )
    return pv.Vinecop.from_data(
        U_vals, structure=cvine_structure, controls=cvine_controls
    )


def vine_diagnostics(U_vals, rng, holdout_weeks=(4, 6, 8)):
    """
    Checks whether the fitted dependence is real rather than noise:
    - null_loglik: in-sample log-lik of a vine refit on column-permuted U (dependence
      destroyed). The real fit's loglik should clearly exceed it.
    - holdout_loglik: mean log-lik on the last k weeks of a vine fit on the earlier
      ones. Independence scores exactly 0, so negative means the vine generalizes worse
      than independent sampling.
    """
    U_null = np.column_stack(
        [rng.permutation(U_vals[:, k]) for k in range(U_vals.shape[1])]
    )
    null_loglik = fit_sparse_cvine(U_null).loglik(U_null)
    holdout = [
        fit_sparse_cvine(U_vals[:-k]).loglik(U_vals[-k:])
        for k in holdout_weeks
        if U_vals.shape[0] - k >= 5
    ]
    holdout_loglik = float(np.mean(holdout)) if holdout else np.nan
    return null_loglik, holdout_loglik


def run_copula_pipeline(
    GW,
    TEAM_ID,
    N_TOP=30,
    N_SAMPLES=20_000,
    T_LIMIT=None,
    min_history_weeks=1,
    seed=42,
    verbose=True,
    diagnostics=True,
    keep_buy=True,
):
    """
    `T_LIMIT=None` (the default) computes the manager's actual banked free transfers
    via `compute_free_transfers_available` instead of using a fixed constant -- pass an
    explicit int to override.

    `keep_buy=True` gives the history weeks the keep/buy marginals of eq. 3.6
    (fit_keep_buy_calibration); without it, alpha's inclusion probabilities.
    """
    if T_LIMIT is None:
        T_LIMIT = compute_free_transfers_available(TEAM_ID, GW, league_selections)

    rng = np.random.default_rng(seed)
    manager_fits = joblib.load(MANAGER_FITS_PATH.format(GW=GW))[GW]

    # keep/buy calibration (eq. 3.6) on the same history window as below
    keep_buy_calib = {
        position: fit_keep_buy_calibration(
            position, get_league_fit_for_round(LEAGUE_FITS[position], GW), GW - 1
        )
        if keep_buy
        else None
        for position in POSITIONS
    }

    # --- pool M: the N_TOP players by target-week alpha, plus the current squad ---
    alpha_target_maps, owned_target = {}, {}
    pool_records = []
    for position in POSITIONS:
        alpha_map, owned = position_alpha_target(
            TEAM_ID,
            position,
            manager_fits[position],
            get_league_fit_for_round(LEAGUE_FITS[position], GW),
            GW,
        )
        alpha_target_maps[position], owned_target[position] = alpha_map, owned
        ranked = sorted(alpha_map.items(), key=lambda kv: kv[1], reverse=True)
        top_elements = {e for e, _ in ranked[:N_TOP]}
        for e in top_elements | owned:
            pool_records.append({"element": e, "position": position})

    pool_df = (
        pd.DataFrame(pool_records)
        .sort_values(["position", "element"])
        .reset_index(drop=True)
    )
    elements_ordered = pool_df["element"].tolist()
    elements_arr = np.array(elements_ordered)
    position_arr = pool_df["position"].to_numpy()
    M = len(elements_ordered)

    # --- history + overhaul filtering ---
    all_history_rounds = sorted(
        league_selections.loc[
            (league_selections["team_id"] == TEAM_ID)
            & (
                league_selections["round"] < GW - 1
            ),  # & (league_selections['round'] >= 4),
            "round",
        ]
        .unique()
        .tolist()
    )

    # print('all_history_rounds: ', all_history_rounds)

    overhaul_rounds = [
        r
        for r in all_history_rounds
        if detect_squad_overhaul(TEAM_ID, r, league_selections)[1]
    ]
    history_rounds = [r for r in all_history_rounds if r not in overhaul_rounds]
    # print('history_rounds: ', history_rounds)
    if len(history_rounds) < min_history_weeks:
        if verbose:
            print(
                f"GW{GW} team {TEAM_ID}: only {len(history_rounds)} non-overhaul history weeks "
                f"(< {min_history_weeks}), skipping"
            )
        return None

    # --- data augmentation ---
    X_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    P_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        pos_elements = pool_df.loc[pool_df["position"] == position, "element"].tolist()
        for rnd in history_rounds:
            p_map, owned_map = position_marginals_at_week(
                TEAM_ID,
                position,
                manager_fit,
                league_fit,
                rnd,
                rng,
                calib=keep_buy_calib[position],
            )
            for e in pos_elements:
                X_hist.loc[rnd, e] = (owned_map or {}).get(e, 0.0)
                P_hist.loc[rnd, e] = p_map.get(e, 1e-6)
    U_vals = latent_uniforms(X_hist, P_hist, rng).values

    # --- sparse C-vine over informative players only ---
    # A player never owned in the history window has u_{t,k} ~ Uniform[0, 1-p) every
    # week: pure jitter with no co-selection signal, so it's left out of the vine and
    # simulated independently below.
    informative_idx = np.where(X_hist.values.sum(axis=0) > 0)[0]
    vine, null_loglik, holdout_loglik = None, np.nan, np.nan
    n_dep_pairs, aic, vine_loglik = 0, 0.0, 0.0
    if len(informative_idx) >= 2:
        U_info = U_vals[:, informative_idx]
        vine = fit_sparse_cvine(U_info)
        n_dep_pairs = sum(
            f != pv.BicopFamily.indep for tree in vine.families for f in tree
        )
        aic = vine.aic(U_info)
        vine_loglik = vine.loglik(U_info)
        if diagnostics:
            null_loglik, holdout_loglik = vine_diagnostics(U_info, rng)

    # --- target-week alpha / ownership / value ---
    alpha_target, current_owned = {}, {}
    for position in POSITIONS:
        for e in pool_df.loc[pool_df["position"] == position, "element"]:
            alpha_target[e] = alpha_target_maps[position].get(e, 1e-12)
            current_owned[e] = float(e in owned_target[position])

    alpha_vec = np.array([alpha_target[e] for e in elements_ordered])
    current_vec = np.array([current_owned[e] for e in elements_ordered])
    position_arr = pool_df["position"].to_numpy()

    value_map = squad_prices(GW)
    value_vec = np.array([value_map.get(e, np.nan) for e in elements_ordered])
    current_squad_value = float(current_vec @ value_vec)

    # --- sampling + constrained per-position selection ---
    sim_U = rng.uniform(size=(N_SAMPLES, M))
    if vine is not None:
        sim_U[:, informative_idx] = vine.simulate(n=N_SAMPLES, seeds=[seed])
    w_star = np.zeros_like(sim_U, dtype=int)
    for position, k in SQUAD_SLOTS.items():
        idxs = np.where(position_arr == position)[0]
        w = np.clip(alpha_vec[idxs], 1e-12, None)
        keys = sim_U[:, idxs] ** (1.0 / w)[None, :]
        topk = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
        rows = np.repeat(np.arange(sim_U.shape[0]), k)
        cols = idxs[topk.ravel()]
        w_star[rows, cols] = 1

    transfers = 0.5 * np.abs(w_star - current_vec[None, :]).sum(axis=1)
    squad_values = w_star @ value_vec
    transfer_ok = transfers <= T_LIMIT
    budget_ok = squad_values <= current_squad_value
    feasible_mask = transfer_ok & budget_ok

    feasible_squads = [
        tuple(elements_arr[row.astype(bool)].tolist()) for row in w_star[feasible_mask]
    ]
    squad_counts = {}
    for sq in feasible_squads:
        squad_counts[sq] = squad_counts.get(sq, 0) + 1

    result = {
        "GW": GW,
        "team_id": TEAM_ID,
        "M": M,
        "n_history_weeks": len(history_rounds),
        "overhaul_rounds": overhaul_rounds,
        "keep_buy_calib": keep_buy_calib,
        "n_informative": len(informative_idx),
        "vine_aic": aic,
        "vine_loglik": vine_loglik,
        "null_loglik": null_loglik,
        "holdout_loglik": holdout_loglik,
        "n_dep_pairs": n_dep_pairs,
        "current_squad_value": current_squad_value,
        "n_samples": N_SAMPLES,
        "T_LIMIT": T_LIMIT,
        "n_transfer_ok": int(transfer_ok.sum()),
        "n_budget_ok": int(budget_ok.sum()),
        "n_feasible": int(feasible_mask.sum()),
        "n_unique_feasible_squads": len(squad_counts),
        "squad_counts": squad_counts,
    }
    if verbose:
        print(
            f"GW{GW:>2} team {TEAM_ID:>8}: M={M:>3}, history={len(history_rounds):>2}wks "
            f"({len(overhaul_rounds)} overhaul), T'={T_LIMIT}, vine dims={len(informative_idx)}, "
            f"dep pairs={n_dep_pairs}, LL={vine_loglik:.1f} (null {null_loglik:.1f}, "
            f"holdout {holdout_loglik:.2f}), "
            f"feasible={result['n_feasible']:>5}/{N_SAMPLES} ({result['n_unique_feasible_squads']} unique)"
        )
    return result

## Points distribution: $\mu_\delta$ and $\Sigma_\delta$

From the predictor's `predictions.csv`: $\mu_\delta$ is the predicted xP at the gameweek. $\Sigma_\delta$ comes from the
residuals (actual − predicted xP) over the earlier gameweeks, plus the points − xP scatter on the diagonal (pooled per
position). Gameweeks without an xP target (22, 32 and 34) have empty residuals and drop out. `SIGMA` picks how the
residuals' correlations enter:

* `"structured"` (default, after Haugh & Singal's Appendix C, `sigma.py`): the standardised residuals of every pair of
  teammates are pooled into one correlation per position pair, fitted by a team factor $\rho(p, q) = a_p a_q$, so
  $\Sigma_\delta = BB^\top + \text{diagonal}$ with one column of $B$ per team. Opposing players' correlations are near zero
  ($|\rho| \le 0.03$) and left out. $\Sigma_\delta$ minus its diagonal has rank at most 20, so the MIQP below keeps all of it,
  with the diagonal as its exact linear term. It explains the next gameweek's points errors better than the per-pair
  version (average Gaussian log-likelihood per player −1.99 against −2.11; −2.01 with no correlations).
* `"sample"`: the per-pair sample covariance of the residuals, projected to the nearest positive definite correlation
  matrix (the saved runs).

In [7]:
def load_predictions(path=PREDICTIONS_PATH):
    """The predictor's predictions, one row per player and gameweek, with `element` ids."""
    return pd.read_csv(path).rename(columns={"fpl_id": "element"})


def sample_mu_sigma_delta(
    GW, preds, min_periods=3, min_eig_frac=1e-4, outcome_noise=True
):
    """
    mu_delta: every player's predicted xP at GW. Sigma_delta: the covariance of their past
    residuals (actual - predicted xP), over pairs with at least `min_periods` gameweeks in
    common; gameweeks 22, 32 and 34 have no xP target and drop out. A player with fewer past residuals (new to the league, or early in the
    season) gets his position's median variance and no covariance; when no player has
    enough yet, the position's residual variance pooled over every past player-gameweek.
    """
    mu_row = preds[preds["round"] == GW].drop_duplicates("element")
    hist = preds[preds["round"] < GW].copy()
    hist["resid"] = hist["xP"] - hist["pred"]
    pivot = hist.pivot_table(index="round", columns="element", values="resid")

    elements = sorted(mu_row["element"])
    rows = mu_row.set_index("element").loc[elements]
    mu_delta = rows["pred"].to_numpy(dtype=float)
    position = rows["pos"].to_numpy()

    cov = pivot.reindex(columns=elements).cov(min_periods=min_periods).to_numpy()
    var = np.diag(cov).copy()
    known = np.isfinite(var)
    for pos in np.unique(position):
        here = position == pos
        if (known & here).any():
            fill = np.median(var[known & here])
        else:
            fill = hist.loc[hist["pos"] == pos, "resid"].var()
        var[~known & here] = fill if np.isfinite(fill) else 1.0
    cov = np.nan_to_num(cov, nan=0.0)
    np.fill_diagonal(cov, np.maximum(var, 1e-4))
    cov = (cov + cov.T) / 2

    std = np.sqrt(np.diag(cov))
    corr = cov / np.outer(std, std)
    np.fill_diagonal(corr, 1.0)

    eigvals, eigvecs = np.linalg.eigh(corr)
    floor = max(min_eig_frac * eigvals.max(), 1e-8)
    corr_psd = eigvecs @ np.diag(np.clip(eigvals, floor, None)) @ eigvecs.T
    d = np.sqrt(np.diag(corr_psd))
    corr_psd = corr_psd / np.outer(d, d)
    np.fill_diagonal(corr_psd, 1.0)
    corr_psd = (corr_psd + corr_psd.T) / 2

    sigma_delta = corr_psd * np.outer(std, std)
    noise_var = np.zeros(len(elements))
    if outcome_noise:
        # The residuals above are xP - pred, but a gameweek is won on points, which scatter
        # around xP by about three times as much (variance 1.9-3.3 against 0.7-1.2). That
        # scatter goes on the diagonal, pooled per position over past player-gameweeks, so
        # it has none of the outliers of per-player points residuals.
        noise = (hist["total_points"] - hist["xP"]).groupby(hist["pos"]).var()
        noise_var = noise.reindex(position).fillna(noise.median()).to_numpy()
        sigma_delta[np.diag_indices_from(sigma_delta)] += noise_var
    # noise_var separately too: the MIQP takes it as an exact linear term
    return elements, mu_delta, sigma_delta, noise_var


import sigma as structured_sigma


def build_mu_sigma_delta(GW, preds):
    """(elements, mu_delta, Sigma_delta, diag_var) with the Sigma chosen by SIGMA."""
    if SIGMA == "structured":
        return structured_sigma.build_mu_sigma_delta(GW, preds)
    return sample_mu_sigma_delta(GW, preds)


FORMATION_MIN = {"Goalkeeper": 1, "Defender": 3, "Midfielder": 2, "Forward": 1}
N_STARTERS = 11


def positions_as_of(gw):
    """{element: position} from each player's latest row at or before `gw`, so players
    without a fixture that round (blank gameweeks) still have a position."""
    latest = (
        player_data[player_data["round"] <= gw]
        .sort_values("round")
        .drop_duplicates("element", keep="last")
    )
    return dict(zip(latest["element"], latest["position"]))


def valid_formation(starters, pos_map):
    counts = {pos: 0 for pos in FORMATION_MIN}
    for e in starters:
        counts[pos_map[e]] += 1
    return counts["Goalkeeper"] == 1 and all(
        counts[pos] >= k for pos, k in FORMATION_MIN.items() if pos != "Goalkeeper"
    )


def pick_lineup(squad, mu_map, pos_map):
    """
    Starting XI, captain, vice-captain and bench for a 15-man squad, chosen on predicted
    points: the best goalkeeper, the formation minimums (3 DEF, 2 MID, 1 FWD) by
    predicted points, then the highest-predicted remaining outfield players; captain and
    vice are the two highest-predicted starters. The data has no real line-ups, so the
    same rule is applied to every squad (ours, the opponent's sampled and real squads).
    The bench is in auto-sub order: reserve goalkeeper, then outfield by predicted points.
    Returns (starters, captain, vice, bench).
    """

    def by_mu(players):
        return sorted(players, key=lambda e: -mu_map.get(e, 0.0))

    by_pos = {
        pos: by_mu([e for e in squad if pos_map.get(e) == pos]) for pos in FORMATION_MIN
    }
    starters = by_pos["Goalkeeper"][:1]
    others = []
    for pos in ["Defender", "Midfielder", "Forward"]:
        starters += by_pos[pos][: FORMATION_MIN[pos]]
        others += by_pos[pos][FORMATION_MIN[pos] :]
    others = by_mu(others)
    n_free = N_STARTERS - len(starters)
    starters += others[:n_free]
    bench = by_pos["Goalkeeper"][1:] + others[n_free:]
    captain, vice = by_mu(starters)[:2]
    return starters, captain, vice, bench


def lineup_weights(lineup):
    """{element: points multiplier}: 1 per starter, 2 for the captain."""
    starters, captain, _, _ = lineup
    weights = {e: 1.0 for e in starters}
    weights[captain] = 2.0
    return weights


def lineup_actuals(lineup, gw_rows, pos_map):
    """
    Actual (xP, total_points) for a line-up under FPL's scoring: the XI, the captain
    doubled, and automatic substitutions -- a starter with 0 minutes is replaced by the
    first bench player (in bench order) who played and keeps a valid formation; if the
    captain didn't play, the vice-captain is doubled instead (if they played).
    `gw_rows` is that round's player rows indexed by element; a player with no row
    (no fixture) counts as not having played.
    """
    starters, captain, vice, bench = lineup

    def played(e):
        return e in gw_rows.index and gw_rows.at[e, "minutes"] > 0

    xi, bench = list(starters), list(bench)
    for e in starters:
        if played(e):
            continue
        for b in bench:
            trial = [b if s == e else s for s in xi]
            if played(b) and valid_formation(trial, pos_map):
                xi = trial
                bench.remove(b)
                break

    weights = {e: 1.0 for e in xi}
    armband = next((c for c in (captain, vice) if c in weights and played(c)), None)
    if armband is not None:
        weights[armband] = 2.0
    rows = gw_rows.reindex(list(weights))
    w = np.array(list(weights.values()))
    return (
        float(rows["xP"].fillna(0.0).to_numpy() @ w),
        float(rows["total_points"].fillna(0.0).to_numpy() @ w),
    )


preds = load_predictions()
print(f"predictions for gameweeks {preds['round'].min()}-{preds['round'].max()}")

predictions for gameweeks 5-38


## Mean-variance MIQP (SCIP)

The model and its constraints (15 players, slots per position, budget, free transfers) are built
once per gameweek; only the objective changes with $\lambda$. Only the starting XI scores, with the
captain doubled, so the model also picks starters $x \le w$ (11, in a valid formation) and a captain
$c \le x$, and scores the multipliers $v = x + c$: mean $\mu^\top v$, variance $v^\top\Sigma v$.

SCIP needs a linear objective, so the variance $v^\top\Sigma v$ goes into an auxiliary
variable $q$ through one quadratic constraint, whose direction depends on the branch:

* `sense=+1` (we are ahead on average): maximise mean − λ·variance with $v^\top\Sigma v \le q$.
  The objective pushes $q$ down onto the variance, and the problem is a convex MIQP.
* `sense=-1` (we are behind): maximise mean + λ·variance with $q \le v^\top\Sigma v$. This is
  nonconvex; SCIP solves it globally, and `time_limit` returns the best squad found so far.
The variance goes through $\Sigma$'s eigendecomposition, $v^\top\Sigma v = \sum_k e_k (u_k^\top v)^2$,
keeping the components above 1% of the largest eigenvalue and bounding the rest from above. Written
pair by pair it has tens of thousands of product terms and SCIP times out at every $\lambda$; this
way each solve is proven optimal in seconds on the convex branch.


In [8]:
def build_mean_variance_model(
    mu_u,
    Sigma_u,
    sigma_delta_G_r,
    price_vec,
    position_vec,
    owned_vec,
    budget,
    T_limit,
    sense=1,
    time_limit=30,
    eig_tol=1e-2,
    diag_var=None,
):
    """
    Builds the SCIP model, constraints, and the (lambda-independent) linear expressions
    once; solve_for_lambda only swaps the linear objective per lambda.

    Only the starting XI scores, with the captain doubled, so besides the squad w the
    model picks starters x (x <= w, 11 of them, 1 GK and at least 3 DEF / 2 MID / 1 FWD)
    and a captain c (c <= x, exactly one). Points come from the multipliers
    v = x + c in {0, 1, 2}: mean mu'v, variance v'Sigma v, covariance with G^{r'} sigma'v.

    The variance is held in the auxiliary variable q (see the markdown above):
    variance <= q for sense=+1, q <= variance for sense=-1. Written directly, v'Sigma v
    has a product term for every pair of players (~45k at P=300), which SCIP can't solve
    to optimality within its time limit even at lambda=0. Instead it goes through
    Sigma's eigendecomposition: v'Sigma v = sum_k e_k (u_k'v)^2, with y_k = u_k'v as
    linear constraints, keeping the components with e_k > eig_tol * max(e) (~45, since
    the residuals come from a few dozen gameweeks). The rest is bounded by
    e_{k+1} * sum_i v_i^2 = e_{k+1} * sum_i (x_i + 3 c_i) (v_i^2 = x_i + 3 c_i because
    c <= x, both binary), so the variance is never understated. This only shapes which
    candidate w(lambda) are generated; each is still scored by Monte Carlo on the full
    Sigma.

    `diag_var` is the diagonal part of Sigma (build_mu_sigma_delta's points - xP noise).
    It is left out of the eigendecomposition, where it would lift every eigenvalue over
    the cut-off (about 300 components instead of 45, and SCIP slows to an hour a
    gameweek), and added exactly instead: sum_i d_i v_i^2 = sum_i d_i (x_i + 3 c_i), linear.

    `time_limit` (seconds, per solve) caps the nonconvex sense=-1 branch, which can run
    for minutes without a cap; SCIP then returns its best incumbent instead of a proof
    of global optimality.
    """
    P = len(mu_u)
    m = Model("mean_variance")
    m.hideOutput()
    m.setParam("limits/time", time_limit)
    # SCIP 10's perspective nonlinear handler stops with "error in input data" on the
    # nonconvex branch (sense=-1) and the process can then crash (malloc / segfault). Off,
    # those solves reach optimality; a solve that does returns the same optimum either way.
    m.setParam("nlhdlr/perspective/enabled", False)
    w = [m.addVar(vtype="B", name=f"w_{i}") for i in range(P)]
    x = [m.addVar(vtype="B", name=f"x_{i}") for i in range(P)]
    c = [m.addVar(vtype="B", name=f"c_{i}") for i in range(P)]
    v = [m.addVar(vtype="I", lb=0, ub=2, name=f"v_{i}") for i in range(P)]
    q = m.addVar(lb=0.0, ub=None, name="variance")

    diag_var = np.zeros(P) if diag_var is None else np.asarray(diag_var, dtype=float)
    eigvals, eigvecs = np.linalg.eigh(Sigma_u - np.diag(diag_var))
    order = np.argsort(-eigvals)
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    n_comp = int((eigvals > eig_tol * eigvals[0]).sum())
    residual_bound = max(eigvals[n_comp], 0.0) if n_comp < P else 0.0
    y = [m.addVar(lb=None, ub=None, name=f"y_{k}") for k in range(n_comp)]

    # squad
    m.addCons(quicksum(w) == 15)
    for pos, k in SQUAD_SLOTS.items():
        idxs = np.where(position_vec == pos)[0]
        m.addCons(quicksum(w[i] for i in idxs) == k)
    m.addCons(quicksum(price_vec[i] * w[i] for i in range(P)) <= budget)
    m.addCons(quicksum(w[i] * (1 - owned_vec[i]) for i in range(P)) <= T_limit)

    # starting XI and captain
    for i in range(P):
        m.addCons(x[i] <= w[i])
        m.addCons(c[i] <= x[i])
        m.addCons(v[i] == x[i] + c[i])
    m.addCons(quicksum(x) == N_STARTERS)
    m.addCons(quicksum(c) == 1)
    for pos, k in FORMATION_MIN.items():
        idxs = np.where(position_vec == pos)[0]
        starters_pos = quicksum(x[i] for i in idxs)
        m.addCons(starters_pos == k if pos == "Goalkeeper" else starters_pos >= k)

    for k in range(n_comp):
        m.addCons(y[k] == quicksum(eigvecs[i, k] * v[i] for i in range(P)))
    variance = quicksum(eigvals[k] * y[k] * y[k] for k in range(n_comp))
    variance += quicksum(
        (residual_bound + diag_var[i]) * (x[i] + 3 * c[i]) for i in range(P)
    )
    if sense == 1:
        m.addCons(variance <= q)
    else:
        m.addCons(q <= variance)

    lin = quicksum(mu_u[i] * v[i] for i in range(P))
    lin_cov = quicksum(sigma_delta_G_r[i] * v[i] for i in range(P))
    return m, (w, v), lin, q, lin_cov


def solve_for_lambda(m, wv, lin, q, lin_cov, P, lam, sense=1):
    """sense=+1 for the mu_Yw >= 0 branch (maximize mean - lam*var), -1 otherwise
    (maximize mean + lam*var); `sense` must match the one the model was built with.
    Returns (w, v): the optimal (or, under the time limit, best-found-so-far) binary
    squad w and points multipliers v (0 bench, 1 starter, 2 captain), each len P, or
    None if no feasible solution was found at all. Checks the number of solutions
    rather than the status, since a time-limit stop with a good incumbent is still
    usable."""
    w, v = wv
    m.freeTransform()  # back to the problem stage, so the objective can be replaced
    m.setObjective(lin - sense * lam * (q - 2 * lin_cov), "maximize")
    m.optimize()
    if m.getNSols() == 0:
        return None
    sol = m.getBestSol()
    return (
        np.array([m.getSolVal(sol, w[i]) for i in range(P)]),
        np.array([m.getSolVal(sol, v[i]) for i in range(P)]),
    )


## One gameweek

Opponent squads → $\mu_\delta$/$\Sigma_\delta$ → our budget, free transfers and player universe → Monte
Carlo of the opponent's $r$-th best squad $G^{r'}$ → MIQP over the $\lambda$ grid → hold or transfer.
`mip_time_limit` is SCIP's time limit per $\lambda$, in seconds.

`method="mvt"` solves the whole $\lambda$ grid and keeps the squad with the highest $P(\text{beat})$.
`method="base"` solves $\lambda = 0$ only (most expected points; the convex branch, whatever the
opponent) and transfers when that adds more than `xp_margin` expected points to holding. Its
$P(\text{beat})$ is computed the same way, for comparison.

Every squad is scored like a real H2H score: the starting XI with the captain doubled. The data
has no real line-ups, so `pick_lineup` picks the XI and captain on predicted points for every
opponent squad in $\mathbb{W}_o$, for holding our current squad, and for the real-squad
baselines; a transfer uses the MIQP's own XI and captain. Actual points add FPL's automatic
substitutions and the vice-captain rule (`lineup_actuals`).

`use_fixture_tilt` needs an `apply_fixture_tilt(elements, mu_delta, GW, n_fixtures,
tilt_strength)` function, which is not defined yet, so it is off by default.

In [9]:
def run_optimization_pipeline(
    GW,
    OWN_TEAM_ID,
    OPPONENT_TEAM_ID,
    R_PCT=0.5,
    N_SAMPLES_OPP=20_000,
    N_MC=2000,
    lambda_grid=None,
    min_resid_rounds=1,
    seed=42,
    verbose=True,
    own_current_squad_override=None,
    own_T_limit_override=None,
    own_bank=0.0,
    transfer_margin=0.03,
    mip_time_limit=30,
    use_fixture_tilt=False,
    fixture_tilt_n=3,
    fixture_tilt_strength=0.05,
    method="mvt",
    xp_margin=0.01,
):
    if method not in ("base", "mvt"):
        raise ValueError(f"method must be 'base' or 'mvt', got {method!r}")
    if method == "base":
        lambda_grid = np.array([0.0])
    elif lambda_grid is None:
        lambda_grid = np.concatenate([[0.0], np.geomspace(0.01, 5.0, 10)])

    # --- Step 1: opponent Wo, falling back to their actual GW-1 squad if empty ---
    opp_result = run_copula_pipeline(
        GW,
        OPPONENT_TEAM_ID,
        N_SAMPLES=N_SAMPLES_OPP,
        seed=seed,
        verbose=False,
        diagnostics=False,
    )
    used_fallback_benchmark = (
        opp_result is None or opp_result["n_unique_feasible_squads"] == 0
    )
    if used_fallback_benchmark:
        opp_prev_squad = squad_players(OPPONENT_TEAM_ID, GW - 1)
        if opp_prev_squad is None:
            if verbose:
                print(
                    f"GW{GW}: opponent Wo empty AND no opponent GW{GW - 1} squad row, skipping"
                )
            return None
        opp_current_squad = tuple(sorted(opp_prev_squad))
        opp_squad_counts = {opp_current_squad: 1}
        if verbose:
            print(
                f"GW{GW}: opponent Wo empty/unavailable -- falling back to their actual "
                f"GW{GW - 1} squad as a single-squad benchmark"
            )
    else:
        opp_squad_counts = opp_result["squad_counts"]

    # --- Step 2: mu_delta / Sigma_delta, guarding on residual history depth ---
    n_hist_rounds = preds.loc[preds["round"] < GW, "round"].nunique()
    if n_hist_rounds < min_resid_rounds:
        if verbose:
            print(
                f"GW{GW}: only {n_hist_rounds} residual-history rounds (< {min_resid_rounds}), skipping"
            )
        return None
    delta_elements, mu_delta, Sigma_delta, noise_delta = build_mu_sigma_delta(GW, preds)
    if use_fixture_tilt:
        mu_delta = apply_fixture_tilt(
            delta_elements,
            mu_delta,
            GW,
            n_fixtures=fixture_tilt_n,
            tilt_strength=fixture_tilt_strength,
        )

    # --- Step 3: own squad, budget, universe ---
    # `own_current_squad_override`/`own_T_limit_override` let a caller (e.g. the rolling
    # season simulation below) inject a *simulated* squad/FT-bank state in place of the
    # real historical one -- everything downstream treats it identically either way.
    if own_current_squad_override is not None:
        own_current_squad = set(own_current_squad_override)
    else:
        own_current_squad = squad_players(OWN_TEAM_ID, GW - 1)
        if own_current_squad is None:
            if verbose:
                print(f"GW{GW}: no own GW{GW - 1} squad row, skipping")
            return None

    gw_players_ = (
        player_data[player_data["round"] == GW]
        .drop_duplicates("element")
        .set_index("element")
    )
    # latest row at or before GW: prices and positions for players without a fixture
    # this round (blank gameweeks), who have no GW row and no prediction
    latest_ = (
        player_data[player_data["round"] <= GW]
        .sort_values("round")
        .drop_duplicates("element", keep="last")
        .set_index("element")
    )
    predicted_ = sorted(set(delta_elements) & set(gw_players_.index))
    # our own players without a fixture stay in the universe (mu = 0), so holding them
    # is still an option
    no_fixture_own_ = sorted(own_current_squad - set(predicted_))
    universe_ = predicted_ + no_fixture_own_
    if len(universe_) < 15:
        if verbose:
            print(f"GW{GW}: player universe too small ({len(universe_)}), skipping")
        return None

    # FPL budget: what the squad is worth plus the money in the bank. Without the bank,
    # every downgrade or price drop would shrink the budget for good.
    own_budget_ = (
        float(latest_.loc[sorted(own_current_squad), "value"].sum()) + own_bank
    )
    own_T_limit_ = (
        own_T_limit_override
        if own_T_limit_override is not None
        else compute_free_transfers_available(OWN_TEAM_ID, GW, league_selections)
    )

    price_vec_ = latest_.loc[universe_, "value"].to_numpy(dtype=float)
    position_vec_ = latest_.loc[universe_, "position"].to_numpy()
    owned_vec_ = np.array([1.0 if e in own_current_squad else 0.0 for e in universe_])

    delta_index_ = {e: i for i, e in enumerate(delta_elements)}
    u_idx_ = np.array([delta_index_[e] for e in predicted_])
    n_extra_ = len(no_fixture_own_)
    mu_u_ = np.concatenate([mu_delta[u_idx_], np.zeros(n_extra_)])
    Sigma_u_ = np.zeros((len(universe_), len(universe_)))
    Sigma_u_[: len(predicted_), : len(predicted_)] = Sigma_delta[np.ix_(u_idx_, u_idx_)]
    Sigma_u_[len(predicted_) :, len(predicted_) :] = 1e-4 * np.eye(n_extra_)
    noise_u_ = np.concatenate([noise_delta[u_idx_], np.zeros(n_extra_)])
    P_ = len(universe_)

    # line-ups are picked on predicted points (no real line-ups in the data); players
    # outside the universe have no prediction and count as 0
    mu_map_ = dict(zip(universe_, mu_u_))
    pos_map_ = positions_as_of(GW)

    # --- Step 4: Monte Carlo (delta, G^{r'}) ---
    rng_ = np.random.default_rng(seed)
    opp_element_index_ = {e: i for i, e in enumerate(universe_)}
    Wo_rows_, Wo_weights_ = [], []
    for squad, count in opp_squad_counts.items():
        # only the opponent's XI scores, captain doubled: G_o = s_o' delta. A player
        # outside the universe has no fixture this round and scores 0.
        row = np.zeros(P_)
        for e, weight in lineup_weights(pick_lineup(squad, mu_map_, pos_map_)).items():
            if e in opp_element_index_:
                row[opp_element_index_[e]] = weight
        Wo_rows_.append(row)
        Wo_weights_.append(count)
    if len(Wo_rows_) == 0:
        if verbose:
            print(f"GW{GW}: no opponent squads to benchmark against, skipping")
        return None
    Wo_ = np.array(Wo_rows_)

    r_index_from_top_ = int(np.ceil((1 - R_PCT) * Wo_.shape[0]))
    r_index_from_top_ = min(max(r_index_from_top_, 1), Wo_.shape[0])

    delta_samples_ = rng_.multivariate_normal(mu_u_, Sigma_u_, size=N_MC, method="eigh")
    G_all_ = delta_samples_ @ Wo_.T
    G_sorted_ = np.sort(G_all_, axis=1)
    G_r_ = G_sorted_[:, -r_index_from_top_]
    mu_G_r_ = G_r_.mean()
    sigma_delta_G_r_ = ((delta_samples_ - mu_u_) * (G_r_ - mu_G_r_)[:, None]).mean(
        axis=0
    )

    # --- Steps 5-6: mean-variance MIQP over the lambda grid ---
    # holding scores the current squad's rule-based XI and captain
    current_lineup_ = pick_lineup(sorted(own_current_squad), mu_map_, pos_map_)
    current_v_ = np.zeros(P_)
    for e, weight in lineup_weights(current_lineup_).items():
        if e in opp_element_index_:
            current_v_[opp_element_index_[e]] = weight
    mu_Yw_check_ = float(mu_u_ @ current_v_ - mu_G_r_)
    # lambda = 0 ignores the variance, so the base optimizer always takes the convex branch
    sense_ = 1 if method == "base" or mu_Yw_check_ >= 0 else -1
    m_, w_, lin_, var_, lin_cov_ = build_mean_variance_model(
        mu_u_,
        Sigma_u_,
        sigma_delta_G_r_,
        price_vec_,
        position_vec_,
        owned_vec_,
        own_budget_,
        own_T_limit_,
        sense=sense_,
        time_limit=mip_time_limit,
        diag_var=noise_u_,
    )

    prob_beat_current_ = float((delta_samples_ @ current_v_ - G_r_ > 0).mean())

    best_prob, best_w, best_v, best_lam = -1.0, None, None, None
    for lam in lambda_grid:
        sol = solve_for_lambda(m_, w_, lin_, var_, lin_cov_, P_, lam, sense=sense_)
        if sol is None:
            continue
        w_lam, v_lam = sol
        prob_beat = float((delta_samples_ @ v_lam - G_r_ > 0).mean())
        if prob_beat > best_prob:
            best_prob, best_w, best_v, best_lam = prob_beat, w_lam, v_lam, lam
    if best_w is None:
        if verbose:
            print(f"GW{GW}: no feasible w(lambda) found, skipping")
        return None

    # --- no-transfer hurdle. mvt: only adopt the optimizer's candidate if it clears the
    # current squad's own Prob(beat opponent) by more than `transfer_margin` --
    # MC/estimation noise makes *some* swap look marginally better almost every week
    # even when the true edge is nil. base: adopt it if it adds more than `xp_margin`
    # expected points (exact, no MC noise).
    if method == "base":
        adopt_ = float(mu_u_ @ best_v - mu_u_ @ current_v_) > xp_margin
    else:
        adopt_ = best_prob > prob_beat_current_ + transfer_margin
    if adopt_:
        squad_star_ = [universe_[i] for i in range(P_) if best_w[i] > 0.5]
        made_transfer_, chosen_prob_, chosen_lam_ = True, best_prob, best_lam
        # the MIQP's own XI and captain; vice and bench order by predicted points
        starters = [universe_[i] for i in range(P_) if best_v[i] > 0.5]
        captain = universe_[int(np.argmax(best_v))]
        by_mu = sorted(starters, key=lambda e: -mu_map_.get(e, 0.0))
        vice = next(e for e in by_mu if e != captain)
        rule_bench = pick_lineup(squad_star_, mu_map_, pos_map_)[3]
        bench = [e for e in rule_bench if e not in starters] + [
            e for e in squad_star_ if e not in starters and e not in rule_bench
        ]
        lineup_star_ = (starters, captain, vice, bench)
    else:
        squad_star_ = sorted(own_current_squad)
        made_transfer_, chosen_prob_, chosen_lam_ = False, prob_beat_current_, None
        lineup_star_ = current_lineup_

    # --- ground truth: XI + captain + auto-subs, like a real H2H score ---
    opp_actual_squad_ = squad_players(OPPONENT_TEAM_ID, GW)
    if opp_actual_squad_ is None:
        opp_xp_, opp_pts_ = None, None
    else:
        opp_xp_, opp_pts_ = lineup_actuals(
            pick_lineup(sorted(opp_actual_squad_), mu_map_, pos_map_),
            gw_players_,
            pos_map_,
        )
    cur_xp_, cur_pts_ = lineup_actuals(current_lineup_, gw_players_, pos_map_)
    opt_xp_, opt_pts_ = lineup_actuals(lineup_star_, gw_players_, pos_map_)

    result = {
        "GW": GW,
        "own_team_id": OWN_TEAM_ID,
        "opponent_team_id": OPPONENT_TEAM_ID,
        "method": method,
        "lambda_star": chosen_lam_,
        "prob_beat_opponent": chosen_prob_,
        "prob_beat_opponent_current": prob_beat_current_,
        "made_transfer": made_transfer_,
        "used_fallback_benchmark": used_fallback_benchmark,
        "own_T_limit": own_T_limit_,
        "own_budget": own_budget_,
        "own_bank_after": own_budget_
        - float(latest_.loc[sorted(squad_star_), "value"].sum()),
        "own_current_squad": sorted(own_current_squad),
        "squad_star": squad_star_,
        "lineup_star": {
            "starters": lineup_star_[0],
            "captain": lineup_star_[1],
            "vice": lineup_star_[2],
            "bench": lineup_star_[3],
        },
        "own_current_squad_players": describe_squad(sorted(own_current_squad), GW),
        "squad_star_players": describe_squad(squad_star_, GW),
        "current_xP": cur_xp_,
        "current_points": cur_pts_,
        "optimized_xP": opt_xp_,
        "optimized_points": opt_pts_,
        "opponent_xP": opp_xp_,
        "opponent_points": opp_pts_,
    }
    if verbose:

        def outcome_(a, b):
            return (
                "N/A"
                if a is None or b is None
                else ("WIN" if a > b else ("LOSS" if a < b else "TIE"))
            )

        flag = " [fallback benchmark]" if used_fallback_benchmark else ""
        action = (
            f"lambda*={chosen_lam_:.3f}"
            if made_transfer_
            else "held (no transfer clears the margin)"
        )
        print(
            f"[{method}] GW{GW:>2} vs {OPPONENT_TEAM_ID:>8}: {action} P(beat)={chosen_prob_:.1%}{flag} | "
            f"xP {cur_xp_:.1f}->{opt_xp_:.1f} vs opp {opp_xp_}  [{outcome_(opt_xp_, opp_xp_)}] | "
            f"pts {cur_pts_:.0f}->{opt_pts_:.0f} vs opp {opp_pts_}  [{outcome_(opt_pts_, opp_pts_)}]"
        )
    return result

## Rolling season

Seeds our squad and free-transfer bank from the real squad before `start_gw`, then carries the
simulated state forward, including the money in the bank (starting at 0, since the data has no
real bank), so the budget each week is the squad's value plus the bank. A gameweek that cannot be solved
keeps the squad unchanged.

In [10]:
def run_rolling_optimization(
    start_gw,
    end_gw,
    OWN_TEAM_ID,
    R_PCT=0.5,
    N_SAMPLES_OPP=20_000,
    N_MC=2000,
    lambda_grid=None,
    min_resid_rounds=1,
    seed=42,
    verbose=True,
    transfer_margin=0.03,
    mip_time_limit=30,
    use_fixture_tilt=False,
    fixture_tilt_n=3,
    fixture_tilt_strength=0.05,
    method="mvt",
):
    current_squad = squad_players(OWN_TEAM_ID, start_gw - 1)
    if current_squad is None:
        raise ValueError(
            f"no real own squad row at GW{start_gw - 1} to seed the rollout from"
        )
    ft_bank = compute_free_transfers_available(OWN_TEAM_ID, start_gw, league_selections)
    # money in the bank, carried like the squad: the data has no real bank, so start at 0
    bank = 0.0

    rollout = []
    for gw in range(start_gw, end_gw + 1):
        opp_team = opponents_by_round[gw]
        try:
            res = run_optimization_pipeline(
                gw,
                OWN_TEAM_ID,
                opp_team,
                R_PCT=R_PCT,
                N_SAMPLES_OPP=N_SAMPLES_OPP,
                N_MC=N_MC,
                lambda_grid=lambda_grid,
                min_resid_rounds=min_resid_rounds,
                seed=seed,
                verbose=verbose,
                own_current_squad_override=current_squad,
                own_T_limit_override=ft_bank,
                own_bank=bank,
                transfer_margin=transfer_margin,
                mip_time_limit=mip_time_limit,
                use_fixture_tilt=use_fixture_tilt,
                fixture_tilt_n=fixture_tilt_n,
                fixture_tilt_strength=fixture_tilt_strength,
                method=method,
            )
        except Exception as e:
            if verbose:
                print(f"GW{gw} vs {opp_team} FAILED: {e!r}")
            res = None

        if res is None:
            # couldn't solve this gameweek at all -- carry the simulated squad forward
            # unchanged, same as a real week with no transfer made.
            ft_bank = min(5, ft_bank + 1)
            rollout.append(
                {
                    "GW": gw,
                    "opponent": opp_team,
                    "status": "skipped",
                    "n_transfers": 0,
                    "ft_bank_after": ft_bank,
                    "bank_after": bank,
                    "squad": sorted(current_squad),
                    "squad_players": describe_squad(sorted(current_squad), gw),
                }
            )
            continue

        new_squad = set(res["squad_star"])
        n_transfers = len(new_squad - current_squad)
        transferred_in = describe_squad(sorted(new_squad - current_squad), gw)
        transferred_out = describe_squad(sorted(current_squad - new_squad), gw)
        ft_bank = min(5, (ft_bank - n_transfers) + 1)
        bank = res["own_bank_after"]
        current_squad = new_squad

        rollout.append(
            {
                "GW": gw,
                "opponent": opp_team,
                "status": "ok",
                "used_fallback_benchmark": res["used_fallback_benchmark"],
                "made_transfer": res["made_transfer"],
                "lambda_star": res["lambda_star"],
                "prob_beat_opponent": res["prob_beat_opponent"],
                "n_transfers": n_transfers,
                "ft_bank_after": ft_bank,
                "bank_after": bank,
                "squad": sorted(current_squad),
                "squad_players": describe_squad(sorted(current_squad), gw),
                "transferred_in": transferred_in,
                "transferred_out": transferred_out,
                "sim_current_xP": res["current_xP"],
                "sim_optimized_xP": res["optimized_xP"],
                "opponent_xP": res["opponent_xP"],
                "sim_current_points": res["current_points"],
                "sim_optimized_points": res["optimized_points"],
                "opponent_points": res["opponent_points"],
            }
        )

    return pd.DataFrame(rollout)

### Run

The first gameweek needs a league fit at or before it (the league betas start at the earliest
gameweek in `LEAGUE_FITS`).

The season is run once per optimizer in `METHODS`, in parallel (one worker process each), and
each run is saved to its own `ROLLING_RESULTS_PATH`.


In [ ]:
import multiprocessing
import os

from threadpoolctl import threadpool_limits

ROLLING_START_GW = max(min(fits) for fits in LEAGUE_FITS.values())
ROLLING_END_GW = 38


def rolling_for_method(method):
    """One rolling season with `method`, saved to its own file. Each worker gets an equal
    share of the cores, so the parallel runs don't fight over the same threads."""
    with threadpool_limits(max(1, (os.cpu_count() or 2) // len(METHODS))):
        rolling_df = run_rolling_optimization(
            ROLLING_START_GW,
            ROLLING_END_GW,
            OWN_TEAM_ID,
            R_PCT=0.5,
            N_SAMPLES_OPP=20_000,
            N_MC=2000,
            seed=42,
            method=method,
        )
    joblib.dump(rolling_df, ROLLING_RESULTS_PATH.format(method=method))


# The optimizers run side by side, one forked process each: fork gives every worker this
# notebook's functions and data without pickling them, and each saves its own results
# file, read back below. Their log lines are tagged [base] / [mvt].
fork = multiprocessing.get_context("fork")
workers = {s: fork.Process(target=rolling_for_method, args=(s,)) for s in METHODS}
for worker in workers.values():
    worker.start()
for worker in workers.values():
    worker.join()
failed = [s for s, worker in workers.items() if worker.exitcode != 0]
if failed:
    raise RuntimeError(f"rolling season failed for {failed}; see the log above")

rolling_by_method = {
    s: joblib.load(ROLLING_RESULTS_PATH.format(method=s)) for s in METHODS
}
for method, rolling_df in rolling_by_method.items():
    n_ok_roll = (rolling_df["status"] == "ok").sum()
    n_held = (rolling_df.get("made_transfer") == False).sum()
    print(
        f"{method}: {n_ok_roll} / {len(rolling_df)} gameweeks produced a rolling result "
        f"({n_held} of those held the squad -- no transfer cleared the margin)"
    )


[mvt] GW 6 vs    23881: lambda*=0.000 P(beat)=92.3% | xP 72.1->73.4 vs opp 75.3  [LOSS] | pts 62->63 vs opp 39.0  [WIN]
[mvt] GW 7 vs    38054: lambda*=0.000 P(beat)=77.1% | xP 61.9->64.0 vs opp 67.2  [LOSS] | pts 42->42 vs opp 42.0  [TIE]
[mvt] GW 8 vs    13603: lambda*=0.000 P(beat)=56.2% | xP 58.6->63.9 vs opp 41.699999999999996  [WIN] | pts 58->59 vs opp 32.0  [WIN]
[mvt] GW 9 vs   593532: lambda*=0.000 P(beat)=77.5% | xP 71.2->73.3 vs opp 62.599999999999994  [WIN] | pts 73->70 vs opp 73.0  [LOSS]
[mvt] GW10 vs   749244: lambda*=0.000 P(beat)=76.6% | xP 71.2->72.2 vs opp 50.70000000000001  [WIN] | pts 62->57 vs opp 30.0  [WIN]
[mvt] GW11 vs   696127: lambda*=0.079 P(beat)=96.4% | xP 64.9->71.1 vs opp 60.1  [WIN] | pts 50->47 vs opp 57.0  [LOSS]
[mvt] GW12 vs  4131447: lambda*=0.079 P(beat)=94.5% | xP 54.4->59.1 vs opp 60.8  [LOSS] | pts 45->46 vs opp 55.0  [LOSS]
[mvt] GW13 vs   152245: lambda*=0.000 P(beat)=35.5% | xP 57.5->60.4 vs opp 73.9  [LOSS] | pts 63->57 vs opp 80.0  [LOSS]

## Compare with the real season

Base and MVT side by side with the real season: points, win rates, transfer weeks, and how
well the predicted $P(\text{beat opponent})$ matches the actual results (Brier score; always
predicting 50% scores 0.25).


In [13]:
# rolling_by_method = {  # to reuse saved runs
#     s: joblib.load(ROLLING_RESULTS_PATH.format(method=s)) for s in METHODS
# }


def real_squad_actuals(team_id, gw):
    """A real squad's (xP, total_points) on the same scoring as the rollout: the
    rule-based XI and captain on predicted points, with auto-subs (see pick_lineup)."""
    squad = squad_players(team_id, gw)
    if squad is None:
        return None, None
    gw_players_ = (
        player_data[player_data["round"] == gw]
        .drop_duplicates("element")
        .set_index("element")
    )
    mu_map_ = (
        preds[preds["round"] == gw]
        .drop_duplicates("element")
        .set_index("element")["pred"]
    ).to_dict()
    pos_map_ = positions_as_of(gw)
    return lineup_actuals(
        pick_lineup(sorted(squad), mu_map_, pos_map_), gw_players_, pos_map_
    )


compare_rows = []
for gw in range(ROLLING_START_GW, ROLLING_END_GW + 1):
    real_xp, real_pts = real_squad_actuals(OWN_TEAM_ID, gw)
    opp_xp, opp_pts = real_squad_actuals(opponents_by_round[gw], gw)
    row = {
        "GW": gw,
        "opponent": opponents_by_round[gw],
        "real_points": real_pts,
        "opponent_points": opp_pts,
        "real_xP": real_xp,
        "opponent_xP": opp_xp,
    }
    for method, rolling_df in rolling_by_method.items():
        roll_row = rolling_df.loc[rolling_df["GW"] == gw].iloc[0]
        ok = roll_row["status"] == "ok"
        row[f"status_{method}"] = roll_row["status"]
        row[f"points_{method}"] = roll_row.get("sim_optimized_points") if ok else None
        row[f"xP_{method}"] = roll_row.get("sim_optimized_xP") if ok else None
        row[f"P_beat_{method}"] = roll_row.get("prob_beat_opponent") if ok else None
        row[f"transfer_{method}"] = roll_row.get("made_transfer") if ok else None
    compare_rows.append(row)
compare_df = pd.DataFrame(compare_rows)

# Weeks a rollout couldn't solve have no simulated squad of their own (the trajectory
# carries the prior squad forward), so their points are NaN rather than fabricated and
# they are left out of that method's summary.
summary_rows = []
for method in rolling_by_method:
    solved = compare_df[compare_df[f"status_{method}"] == "ok"]
    pts, opp = solved[f"points_{method}"], solved["opponent_points"]
    win = (pts > opp).astype(float) + 0.5 * (pts == opp)
    p_beat = solved[f"P_beat_{method}"].astype(float)
    summary_rows.append(
        {
            "method": method,
            "gameweeks solved": len(solved),
            "points": pts.sum(),
            "real history points": solved["real_points"].sum(),
            "opponent points": opp.sum(),
            "beat real history": (pts > solved["real_points"]).mean(),
            "win rate vs opponent": win.mean(),
            "mean predicted P(beat)": p_beat.mean(),
            "Brier score": ((p_beat - win) ** 2).mean(),
            "mean xP": solved[f"xP_{method}"].astype(float).mean(),
            "transfer weeks": int(solved[f"transfer_{method}"].sum()),
        }
    )
summary_df = pd.DataFrame(summary_rows).set_index("method")
print(summary_df.round(3).T.to_string())

compare_df


method                       mvt
gameweeks solved          33.000
points                  1805.000
real history points     2027.000
opponent points         1950.000
beat real history          0.303
win rate vs opponent       0.318
mean predicted P(beat)     0.628
Brier score                0.394
mean xP                   63.258
transfer weeks            32.000


,GW,opponent,real_points,opponent_points,real_xP,opponent_xP,status_mvt,points_mvt,xP_mvt,P_beat_mvt,transfer_mvt
0,6,23881,40.0,39.0,71.4,75.3,ok,63.0,73.4,0.9230,True
1,7,38054,51.0,42.0,66.2,67.2,ok,42.0,64.0,0.7710,True
2,8,13603,41.0,32.0,52.5,41.7,ok,59.0,63.9,0.5620,True
3,9,593532,75.0,73.0,66.5,62.6,ok,70.0,73.3,0.7750,True
4,10,749244,66.0,30.0,66.7,50.7,ok,57.0,72.2,0.7660,True
5,11,696127,61.0,57.0,72.4,60.1,ok,47.0,71.1,0.9635,True
6,12,4131447,68.0,55.0,64.2,60.8,ok,46.0,59.1,0.9445,True
7,13,152245,87.0,80.0,71.9,73.9,ok,57.0,60.4,0.3550,True
8,14,1441046,58.0,59.0,59.3,79.7,ok,75.0,66.0,0.1440,True
9,15,1222335,57.0,56.0,65.7,67.8,ok,77.0,56.7,0.0645,True
